# External-Signals medallion -- SIT evidence run

Sprint 21 signal Fabric evidence (Task 4). Deterministically materializes the
bronze / silver / gold `ext_*` tables from the committed synthetic seed so the
`external-signals` semantic model and the `da_hospital_capacity` data agent can
be proven end to end. Synthetic-only, no PHI (ADR-0013 / ADR-0016).

Generated by `data-platform/scripts/fabric/build_ext_evidence_notebook.py` --
do not edit by hand; re-generate to update.


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, ArrayType,
)
from pyspark.sql import functions as F

FLAT_SCHEMA = StructType([
    StructField("ext_signal_id", StringType(), True),
    StructField("ext_source_id", StringType(), True),
    StructField("ext_source_authority", StringType(), True),
    StructField("ext_trust_tier", StringType(), True),
    StructField("ext_hazard_type", StringType(), True),
    StructField("ext_severity", StringType(), True),
    StructField("ext_status", StringType(), True),
    StructField("ext_scenario_template", StringType(), True),
    StructField("ext_lage_tier", LongType(), True),
    StructField("ext_cantons", ArrayType(StringType()), True),
    StructField("ext_onset", StringType(), True),
    StructField("ext_active_binding", StringType(), True),
    StructField("ext_fell_back_from", StringType(), True),
    StructField("ext_ingested_at", StringType(), True),
])

FACT_SCHEMA = StructType([
    StructField("ext_signal_id", StringType(), True),
    StructField("ext_source_id", StringType(), True),
    StructField("ext_hazard_type", StringType(), True),
    StructField("ext_severity", StringType(), True),
    StructField("ext_scenario_template", StringType(), True),
    StructField("ext_lage_tier", LongType(), True),
    StructField("ext_cantons", StringType(), True),
    StructField("ext_onset", StringType(), True),
    StructField("ext_status", StringType(), True),
])

SOURCE_SCHEMA = StructType([
    StructField("ext_source_id", StringType(), True),
    StructField("ext_source_authority", StringType(), True),
    StructField("ext_trust_tier", StringType(), True),
    StructField("ext_data_mode", StringType(), True),
    StructField("ext_fell_back_from", StringType(), True),
    StructField("ext_last_live_at", StringType(), True),
])

HAZARD_SCHEMA = StructType([
    StructField("ext_hazard_type", StringType(), True),
    StructField("ext_scenario_template", StringType(), True),
    StructField("ext_default_lage_tier", LongType(), True),
])

REGION_SCHEMA = StructType([
    StructField("ext_canton", StringType(), True),
])

TRIGGER_SCHEMA = StructType([
    StructField("ext_trigger_event_id", StringType(), True),
    StructField("ext_signal_id", StringType(), True),
    StructField("ext_hazard_type", StringType(), True),
    StructField("ext_severity", StringType(), True),
    StructField("ext_lage_tier", LongType(), True),
    StructField("ext_scenario_template", StringType(), True),
    StructField("ext_sources", StringType(), True),
    StructField("ext_signal_ids", StringType(), True),
    StructField("ext_dedup_keys", StringType(), True),
    StructField("ext_trigger_status", StringType(), True),
    StructField("ext_source_onset", StringType(), True),
    StructField("ext_triggered_at", StringType(), True),
    StructField("ext_run_id", StringType(), True),
])


def _write(rows, schema, table, ts_cols=()):
    # Direct Lake models only consume scalar Delta columns; ISO-string date
    # columns modeled as dateTime are cast to real Spark timestamps on write.
    df = spark.createDataFrame(rows, schema)
    for col in ts_cols:
        df = df.withColumn(col, F.to_timestamp(col))
    df.write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(table)
    print(f"wrote {table}: {df.count()} rows")


In [ ]:
# Bronze -- raw external signals landed verbatim (flat evidence shape)
bronze_rows = [['cap-2026-heat-zh-1', 'alertswiss', 'BABS/FOCP', 'A', 'heat', 'Severe', 'Actual', 'F8', 2, ['ZH'], '2026-07-17T12:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['bag-rsv-2026-w29', 'bag', 'FOPH/BAG', 'A', 'rsv', 'Moderate', 'Actual', 'F6', 2, ['BE', 'ZH'], '2026-07-13T00:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['ms-heat-2026-0001', 'meteoswiss', 'MeteoSwiss', 'A', 'heat', 'Severe', 'Actual', 'F8', 2, ['ZH'], '2026-07-17T12:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['sed-2026-0007', 'sed', 'SED-ETH', 'A', 'earthquake', 'Severe', 'Actual', 'F1', 3, ['VS'], '2026-07-17T10:00:00Z', 'live', None, '2026-07-23T14:47:15Z']]
_write(bronze_rows, FLAT_SCHEMA, "bronze.ext_signals_raw")


In [ ]:
# Silver -- kept Actual signals + (empty) quarantine, same schema
silver_rows = [['cap-2026-heat-zh-1', 'alertswiss', 'BABS/FOCP', 'A', 'heat', 'Severe', 'Actual', 'F8', 2, ['ZH'], '2026-07-17T12:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['bag-rsv-2026-w29', 'bag', 'FOPH/BAG', 'A', 'rsv', 'Moderate', 'Actual', 'F6', 2, ['BE', 'ZH'], '2026-07-13T00:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['ms-heat-2026-0001', 'meteoswiss', 'MeteoSwiss', 'A', 'heat', 'Severe', 'Actual', 'F8', 2, ['ZH'], '2026-07-17T12:00:00Z', 'live', None, '2026-07-23T14:47:15Z'], ['sed-2026-0007', 'sed', 'SED-ETH', 'A', 'earthquake', 'Severe', 'Actual', 'F1', 3, ['VS'], '2026-07-17T10:00:00Z', 'live', None, '2026-07-23T14:47:15Z']]
quarantine_rows = []
_write(silver_rows, FLAT_SCHEMA, "silver.ext_signals")
_write(quarantine_rows, FLAT_SCHEMA, "silver.ext_signals_quarantine")


In [ ]:
# Gold -- star schema consumed by the semantic model + data agent
fact_rows = [['cap-2026-heat-zh-1', 'alertswiss', 'heat', 'Severe', 'F8', 2, 'ZH', '2026-07-17T12:00:00Z', 'Actual'], ['bag-rsv-2026-w29', 'bag', 'rsv', 'Moderate', 'F6', 2, 'BE,ZH', '2026-07-13T00:00:00Z', 'Actual'], ['ms-heat-2026-0001', 'meteoswiss', 'heat', 'Severe', 'F8', 2, 'ZH', '2026-07-17T12:00:00Z', 'Actual'], ['sed-2026-0007', 'sed', 'earthquake', 'Severe', 'F1', 3, 'VS', '2026-07-17T10:00:00Z', 'Actual']]
source_rows = [['alertswiss', 'BABS/FOCP', 'A', 'Live', None, '2026-07-23T14:47:15Z'], ['bag', 'FOPH/BAG', 'A', 'Live', None, '2026-07-23T14:47:15Z'], ['meteoswiss', 'MeteoSwiss', 'A', 'Live', None, '2026-07-23T14:47:15Z'], ['sed', 'SED-ETH', 'A', 'Live', None, '2026-07-23T14:47:15Z']]
hazard_rows = [['earthquake', 'F1', 3], ['heat', 'F8', 2], ['rsv', 'F6', 2]]
region_rows = [['BE'], ['VS'], ['ZH']]
_write(fact_rows, FACT_SCHEMA, "gold.ext_fact_signal", ts_cols=("ext_onset",))
_write(source_rows, SOURCE_SCHEMA, "gold.ext_dim_source", ts_cols=("ext_last_live_at",))
_write(hazard_rows, HAZARD_SCHEMA, "gold.ext_dim_hazard_type")
_write(region_rows, REGION_SCHEMA, "gold.ext_dim_region")


In [ ]:
# Gold -- trigger-event audit fact (collapsed HazardEvents that fire a pre-seed)
trigger_rows = [['trg-heat-0001', 'cap-2026-heat-zh-1', 'heat', 'Severe', 2, 'F8', 'alertswiss,meteoswiss', 'cap-2026-heat-zh-1,ms-heat-2026-0001', 'heat:alertswiss,heat:meteoswiss', 'trigger-fired', '2026-07-17T12:00:00Z', '2026-07-23T14:47:15Z', 'evidence-run'], ['trg-rsv-0002', 'bag-rsv-2026-w29', 'rsv', 'Moderate', 2, 'F6', 'bag', 'bag-rsv-2026-w29', 'rsv:bag', 'trigger-fired', '2026-07-13T00:00:00Z', '2026-07-23T14:47:15Z', 'evidence-run'], ['trg-earthquake-0003', 'sed-2026-0007', 'earthquake', 'Severe', 3, 'F1', 'sed', 'sed-2026-0007', 'earthquake:sed', 'trigger-fired', '2026-07-17T10:00:00Z', '2026-07-23T14:47:15Z', 'evidence-run']]
_write(trigger_rows, TRIGGER_SCHEMA, "gold.ext_fact_trigger_event",
       ts_cols=("ext_source_onset", "ext_triggered_at"))


In [ ]:
# Inline verification -- print counts + distinct data modes for the evidence doc
for t in ["bronze.ext_signals_raw", "silver.ext_signals",
          "silver.ext_signals_quarantine", "gold.ext_fact_signal",
          "gold.ext_dim_source", "gold.ext_dim_hazard_type",
          "gold.ext_dim_region", "gold.ext_fact_trigger_event"]:
    print(t, spark.table(t).count())
display(spark.table("gold.ext_dim_source").select(
    "ext_source_id", "ext_source_authority", "ext_trust_tier", "ext_data_mode"))
